# Full test-set evaluation: ResUNet and data-consistency post-processing


## Prepare


In [1]:
#============= Globals ==========
import torch
from src.general_utils import prepare_environment
from mri_dl import (
    ModelConfig,
    MRIUndersampledDataset,
    ResidualUNet,
    evaluate_and_save_results,
    load_checkpoint,
)
from torch.utils.data import DataLoader
import os
from pathlib import Path
import pandas as pd
HPC = False
#============= main ==========
prepare_environment(hpc=HPC)

experiment_config = ModelConfig(
    plane="coronal",
    retain_ratio=0.30,
    #epochs=60,
    epochs=2,
    batch_size=32,
    learning_rate=5e-4,
    weight_decay=1e-4,
    random_seed=42,
    num_workers=0 if os.name == "nt" else 4,
    data_consistency_enabled=True,
    per_image_csv_logging=True,
    data_root=Path(os.getcwd() + r"/../undersampled_dataset_split").resolve(),
    result_root=Path(os.getcwd() + r"/../undersampled_results").resolve() )


Data path: C:\mri_dataset\brain_age


## Load test dataset and trained model


In [2]:
test_dataset = MRIUndersampledDataset(
    experiment_config.data_root / "test",
    csv_name="samples.csv",
    plane=experiment_config.plane.capitalize(),
    retain_ratio=experiment_config.retain_ratio,
    load_mask=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)
loaded_model, checkpoint = load_checkpoint(
    checkpoint_path=experiment_config.result_dir / "best_model.pt",
    model_class=ResidualUNet,
)
print(f"Test samples: {len(test_dataset)}")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["validation_loss"])


Test samples: 200
Best epoch: 51
Best validation loss: 0.024740325659513472


## Evaluate the full test set
This step saves per-image results to `test_per_image.csv`.


In [3]:
if len(test_dataset) == 0:
    raise ValueError("test_dataset is empty")
per_sample_df = evaluate_and_save_results(
    model=loaded_model,
    test_loader=test_loader,
    output_root=Path("results"),
    plane="Coronal",
    retain_ratio=0.30,
    checkpoint_name="coronal_r30_delta_best.pt",
    epoch=checkpoint.get("epoch"),
)
display(per_sample_df.head())
print(per_sample_df.shape)


Saved rows: 200
Output path: results\coronal\retain_30\test_per_image.csv
Unique volumes: 50
Plane: Coronal
Retain ratio: 0.3


,sample_id,volume_id,plane,slice_index,retain_ratio,mask_id,masked_rows,psnr_undersampled,psnr_resunet,psnr_resunet_dc,...,resunet_improved_ssim,dc_improved_ssim,final_improved_ssim_vs_us,checkpoint_name,epoch,split,normalization_method,data_range,metric_policy,inference_time_ms
0,3668_row000276_coronal_s0042_r30_u000,3668,Coronal,42,0.3,mask_3668_row000276_coronal_s0042_r30_u000,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14,...",21.549314,23.595438,24.850283,...,True,False,True,coronal_r30_delta_best.pt,51,test,clamp_to_unit_range,"reference_min_max (PSNR/SSIM), no per-image re...","PSNR_ROI(reference>0), SSIM_full_frame",0.3045
1,3668_row000276_coronal_s0042_r30_u001,3668,Coronal,42,0.3,mask_3668_row000276_coronal_s0042_r30_u001,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14,...",22.116484,24.405414,25.589631,...,True,False,True,coronal_r30_delta_best.pt,51,test,clamp_to_unit_range,"reference_min_max (PSNR/SSIM), no per-image re...","PSNR_ROI(reference>0), SSIM_full_frame",0.3101
2,3668_row000276_coronal_s0055_r30_u000,3668,Coronal,55,0.3,mask_3668_row000276_coronal_s0055_r30_u000,"[0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14...",20.836053,22.943399,24.584367,...,True,False,True,coronal_r30_delta_best.pt,51,test,clamp_to_unit_range,"reference_min_max (PSNR/SSIM), no per-image re...","PSNR_ROI(reference>0), SSIM_full_frame",0.3122
3,3668_row000276_coronal_s0055_r30_u001,3668,Coronal,55,0.3,mask_3668_row000276_coronal_s0055_r30_u001,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",21.337256,24.043973,25.187324,...,True,False,True,coronal_r30_delta_best.pt,51,test,clamp_to_unit_range,"reference_min_max (PSNR/SSIM), no per-image re...","PSNR_ROI(reference>0), SSIM_full_frame",0.3074
4,3852_row001685_coronal_s0043_r30_u000,3852,Coronal,43,0.3,mask_3852_row001685_coronal_s0043_r30_u000,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",22.698472,24.614354,25.325652,...,True,False,True,coronal_r30_delta_best.pt,51,test,clamp_to_unit_range,"reference_min_max (PSNR/SSIM), no per-image re...","PSNR_ROI(reference>0), SSIM_full_frame",0.3140


(200, 32)


## Summary table (mean/std)


In [4]:
summary_rows = [
    {
        "Method": "undersampled image",
        "Mean PSNR": per_sample_df["psnr_undersampled"].mean(),
        "STD PSNR": per_sample_df["psnr_undersampled"].std(ddof=1),
        "Mean SSIM": per_sample_df["ssim_undersampled"].mean(),
        "STD SSIM": per_sample_df["ssim_undersampled"].std(ddof=1),
    },
    {
        "Method": "ResUnet results",
        "Mean PSNR": per_sample_df["psnr_resunet"].mean(),
        "STD PSNR": per_sample_df["psnr_resunet"].std(ddof=1),
        "Mean SSIM": per_sample_df["ssim_resunet"].mean(),
        "STD SSIM": per_sample_df["ssim_resunet"].std(ddof=1),
    },
    {
        "Method": "ResUnet+data consistency (post processing) results",
        "Mean PSNR": per_sample_df["psnr_resunet_dc"].mean(),
        "STD PSNR": per_sample_df["psnr_resunet_dc"].std(ddof=1),
        "Mean SSIM": per_sample_df["ssim_resunet_dc"].mean(),
        "STD SSIM": per_sample_df["ssim_resunet_dc"].std(ddof=1),
    },
]
summary_df = pd.DataFrame(summary_rows).set_index("Method")
display(summary_df.style.format(precision=4))


,Mean PSNR,STD PSNR,Mean SSIM,STD SSIM
Method,,,,
undersampled image,23.2902,1.7028,0.6627,0.0436
ResUnet results,25.8499,1.8845,0.8979,0.0324
ResUnet+data consistency (post processing) results,27.3629,1.9045,0.8296,0.0291


## Average deltas per step


In [5]:
delta_df = pd.DataFrame(
    [
        {
            "Step": "delta_CNN (ResUnet - undersampled)",
            "Mean Delta PSNR": per_sample_df["psnr_gain_resunet_vs_us"].mean(),
            "Mean Delta SSIM": per_sample_df["ssim_gain_resunet_vs_us"].mean(),
        },
        {
            "Step": "delta_consistency (ResUnet+DC - ResUnet)",
            "Mean Delta PSNR": per_sample_df["psnr_gain_dc_vs_resunet"].mean(),
            "Mean Delta SSIM": per_sample_df["ssim_gain_dc_vs_resunet"].mean(),
        },
    ]
).set_index("Step")
display(delta_df.style.format(precision=4))


,Mean Delta PSNR,Mean Delta SSIM
Step,,
delta_CNN (ResUnet - undersampled),2.5596,0.2352
delta_consistency (ResUnet+DC - ResUnet),1.5130,-0.0682


## Post-processing outcome counts


In [6]:
total = len(per_sample_df)
psnr_improved = per_sample_df["dc_improved_psnr"]
ssim_improved = per_sample_df["dc_improved_ssim"]
psnr_up_ssim_down = per_sample_df["dc_improved_psnr"] & (~per_sample_df["dc_improved_ssim"])
outcome_df = pd.DataFrame(
    [
        {
            "Condition": "Post processing improved PSNR",
            "Count": int(psnr_improved.sum()),
            "Percentage (%)": 100.0 * float(psnr_improved.mean()),
        },
        {
            "Condition": "Post processing improved SSIM",
            "Count": int(ssim_improved.sum()),
            "Percentage (%)": 100.0 * float(ssim_improved.mean()),
        },
        {
            "Condition": "PSNR improved while SSIM reduced",
            "Count": int(psnr_up_ssim_down.sum()),
            "Percentage (%)": 100.0 * float(psnr_up_ssim_down.mean()),
        },
    ]
).set_index("Condition")
print(f"Total evaluated images: {total}")
display(outcome_df.style.format({"Count": "{:.0f}", "Percentage (%)": "{:.2f}"}))


Total evaluated images: 200


,Count,Percentage (%)
Condition,,
Post processing improved PSNR,200,100.00
Post processing improved SSIM,10,5.00
PSNR improved while SSIM reduced,190,95.00
